In [51]:
#STEP 1: IMPORTING AND GETTING FILE PATH FOR RAW DATA

import json
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter

raw_dir = Path("..") / "data" / "raw"
files = list(sorted(raw_dir.glob("*.jsonl")))
print(files)



[PosixPath('../data/raw/r_ClaudeAI_posts.jsonl'), PosixPath('../data/raw/r_OpenAi_posts.jsonl'), PosixPath('../data/raw/r_artificial_posts.jsonl'), PosixPath('../data/raw/r_technology_posts.jsonl')]


In [55]:
#STEP 2: COUNT POSTS AND DATE RANGE PER FILE

for path in files:
    count = 0
    earliest = float("inf")
    latest = float("-inf")
    
    with open(path) as fh:
        for line in fh:
            post = json.loads(line)
            ts = post["created_utc"]
            count += 1
            earliest = min(earliest, ts)
            latest = max(latest, ts)

    print(path)
    print(count)
    print(datetime.fromtimestamp(earliest, tz = timezone.utc))
    print(datetime.fromtimestamp(latest, tz = timezone.utc))

../data/raw/r_ClaudeAI_posts.jsonl
43219
2024-11-01 00:47:16+00:00
2025-12-30 23:53:06+00:00
../data/raw/r_OpenAi_posts.jsonl
57180
2024-11-01 01:05:19+00:00
2025-12-30 23:57:25+00:00
../data/raw/r_artificial_posts.jsonl
19818
2024-11-01 00:03:10+00:00
2025-12-30 23:31:50+00:00
../data/raw/r_technology_posts.jsonl
51539
2024-11-01 00:15:01+00:00
2025-12-30 23:37:36+00:00


In [56]:
# STEP 3: FIELD NAMES
with open(files[0]) as fh:
    first_line = fh.readline()

first_post = json.loads(first_line)
print(len(first_post))
print(list(first_post.keys()))

# fields related to removal
for key in first_post.keys():
    if "removed" in key:
        print(key, first_post[key])

106
['_meta', 'all_awardings', 'allow_live_comments', 'approved_at_utc', 'approved_by', 'archived', 'author', 'author_flair_background_color', 'author_flair_css_class', 'author_flair_richtext', 'author_flair_template_id', 'author_flair_text', 'author_flair_text_color', 'author_flair_type', 'author_fullname', 'author_is_blocked', 'author_patreon_flair', 'author_premium', 'awarders', 'banned_at_utc', 'banned_by', 'can_gild', 'can_mod_post', 'category', 'clicked', 'content_categories', 'contest_mode', 'created', 'created_utc', 'discussion_type', 'distinguished', 'domain', 'downs', 'edited', 'gilded', 'gildings', 'hidden', 'hide_score', 'id', 'is_created_from_ads_ui', 'is_crosspostable', 'is_meta', 'is_original_content', 'is_reddit_media_domain', 'is_robot_indexable', 'is_self', 'is_video', 'likes', 'link_flair_background_color', 'link_flair_css_class', 'link_flair_richtext', 'link_flair_template_id', 'link_flair_text', 'link_flair_text_color', 'link_flair_type', 'locked', 'media', 'media_

In [57]:
# STEP 4: WHAT TEXT DO WE HAVE, AND WHAT WAS REMOVED?
for path in files:
    total = 0
    link = 0            # selftext == ""  (no body)
    text_missing = 0    # selftext is "[removed]" or "[deleted]"
    author_deleted = 0
    removed = 0         # removed_by_category is set
    categories = Counter()

    with open(path) as fh:
        for line in fh:
            post = json.loads(line)
            total += 1
            
            if post["author"] == "[deleted]":
                author_deleted += 1

            if post["selftext"] == "":
                link += 1
            elif post["selftext"] in ("[removed]", "[deleted]"):
                text_missing += 1

            category = post.get("removed_by_category")
            if category is not None:
                removed += 1
                categories[category] += 1

    print(f"{path.name}: {total} posts")
    print(f"   link posts:     {link / total:.1%}")
    print(f"   text missing:   {text_missing / total:.1%}")
    print(f"   author deleted: {author_deleted / total:.1%}")
    print(f"   removed:        {removed / total:.1%}  {dict(categories)}")

r_ClaudeAI_posts.jsonl: 43219 posts
   link posts:     12.3%
   text missing:   30.4%
   author deleted: 0.5%
   removed:        69.4%  {'moderator': 2056, 'reddit': 9043, 'automod_filtered': 18671, 'deleted': 224, 'content_takedown': 7, 'author': 2}
r_OpenAi_posts.jsonl: 57180 posts
   link posts:     19.0%
   text missing:   47.7%
   author deleted: 1.0%
   removed:        54.3%  {'reddit': 10205, 'automod_filtered': 12462, 'moderator': 7842, 'deleted': 496, 'content_takedown': 21}
r_artificial_posts.jsonl: 19818 posts
   link posts:     28.1%
   text missing:   47.4%
   author deleted: 1.3%
   removed:        59.4%  {'moderator': 1568, 'automod_filtered': 8731, 'reddit': 1223, 'deleted': 239, 'content_takedown': 5}
r_technology_posts.jsonl: 51539 posts
   link posts:     96.7%
   text missing:   3.2%
   author deleted: 2.5%
   removed:        43.4%  {'moderator': 12464, 'reddit': 5331, 'automod_filtered': 3283, 'deleted': 1270, 'content_takedown': 25}


## Findings

- **Size:** 171,756 posts in total: r/OpenAI 57,180, r/technology 51,539, r/ClaudeAI 43,219, r/artificial 19,818. All four files cover 2024-11-01 to 2025-12-30. Dec 31 is missing because the download tool's end date is exclusive, and I drop incomplete weeks at both ends anyway.

- **Text:** r/technology is almost all link posts (96.7%), so for that subreddit the model will only see headlines. r/ClaudeAI has the most body text (only 12.3% link posts), then r/OpenAI (19.0%) and r/artificial (28.1%). The text for each post will be title + selftext.

- **Removals:** a large share of every subreddit was removed, by `removed_by_category`:

  | Subreddit | Removed | Biggest categories |
  |---|---|---|
  | r/ClaudeAI | 69.4% | automod_filtered 18,671, reddit 9,043, moderator 2,056 |
  | r/artificial | 59.4% | automod_filtered 8,731, moderator 1,568, reddit 1,223 |
  | r/OpenAI | 54.3% | automod_filtered 12,462, reddit 10,205, moderator 7,842 |
  | r/technology | 43.4% | moderator 12,464, reddit 5,331, automod_filtered 3,283 |

  This is why my New-tab estimates were so low: normal users never see removed posts, but the archive saved them.

- **`selftext` undercounts removals:** a removed link post keeps an empty selftext, so checking for `[removed]` found only 3.2% removed in r/technology instead of 43.4%. Also, many removed posts still have their original text, because the archive saved them before removal (r/ClaudeAI: 69.4% removed, but only 30.4% with missing text).

- **Open decision:** which removed posts to keep. `reddit` and `automod_filtered` are probably mostly spam. `moderator` removals are unclear, since some could be posts redirected to a megathread during a big event, and dropping them could hide a real trend. Next session: read ~20 random posts per category before deciding.